# ZeroER feature space — t-SNE exploration (manual features vs. PLM embeddings)

manual features vs PLM embeddings

1. **manual sim-features** — the live-pipeline per-pair similarity vector
   (`pyjedai_module._build_feature_matrix`, 6 string sims per attribute);

2. **a ladder of sentence-transformers** (small → large, incl. the 1024-D
   "large", t5-xl and e5-instruct models) — concat every attribute column of a
   record into one string, embed it (on GPU when available), and represent each
   *pair* by `|u − v|` of the two L2-normalised record embeddings.

Each dataset is written out as a static `.png` grid, `and a tidy`.parquet`; the Streamlit app `tsne_dashboard.py` reads the
parquet so you can pick a point and inspect the original record pair column by
column.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))   # ZeroER/ — for utils & pyjedai_module

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sentence_transformers import SentenceTransformer

from utils import DATASETS, load_data
from pyjedai_module import ZeroEREstimator, _build_feature_matrix

# --- device ---------------------------------------------------------------
# Embedding is the expensive step; it runs on the GPU when one is visible and
# silently falls back to CPU otherwise. (t-SNE stays on sklearn/CPU — it is
# seconds on a few-thousand points, not worth a cuML/openTSNE dependency.)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# --- config ---------------------------------------------------------------
DATASETS_TO_RUN = ["abt_buy", "amazon_googleproducts"]   # the two hard ones

# short name -> (HF sentence-transformer id, encode-time text prefix), ordered
# small -> large. The prefix is empty for everything except the e5 family,
# which is trained asymmetrically: prepending "query: " (or the instruct
# template) to both records is the documented way to use it for symmetric
# similarity. The 1024-D "large" + the t5-xl / e5-instruct models are the
# heavy ones — they really want the GPU.
E5_INSTRUCT = "Instruct: Retrieve semantically similar text.\nQuery: "
PLM_MODELS = {
    "minilm6":       ("sentence-transformers/all-MiniLM-L6-v2",     ""),
    "minilm12":      ("sentence-transformers/all-MiniLM-L12-v2",    ""),
    "distilroberta": ("sentence-transformers/all-distilroberta-v1", ""),
    "mpnet":         ("sentence-transformers/all-mpnet-base-v2",    ""),
    "gtr_t5_large":  ("sentence-transformers/gtr-t5-large",         ""),
    "bge_large":     ("BAAI/bge-large-en-v1.5",                     ""),
    "gte_large":     ("thenlper/gte-large",                         ""),
    "e5_large_v2":   ("intfloat/e5-large-v2",                       "query: "),
    "st5_xl":        ("sentence-transformers/sentence-t5-xl",       ""),
    "e5_instruct":   ("intfloat/multilingual-e5-large-instruct",    E5_INSTRUCT),
}
N_NEGATIVES = 6000          # non-matches embedded (all matches are always kept)
BLOCKER     = "standard"    # will experiment with PLM blocker as well at some point
SEED        = 0
OUT_DIR = os.path.abspath(os.path.join("..", "outputs", "tsne_notebook"))
os.makedirs(OUT_DIR, exist_ok=True)
print("cwd:", os.getcwd())
print("device:", DEVICE)
print("saving figures + exports to:", OUT_DIR)

## 1. Candidate pairs + gold labels

Positives = all gold matches. Negatives = the *hard* non-matches that survive
the blocker (what the matcher actually has to reject), subsampled to
`N_NEGATIVES`. The same pair set feeds all four panels, so they're comparable.

In [2]:
def gold_pairs(data):
    """Gold matches as positional (left_idx, right_idx) tuples into data.entities."""
    out = []
    for _, (lid, rid) in data.ground_truth.iterrows():
        if lid in data._ids_mapping_1 and rid in data._ids_mapping_2:
            out.append((data._ids_mapping_1[lid], data._ids_mapping_2[rid]))
    return out


def build_pairs_and_labels(data, n_negatives=N_NEGATIVES, blocker=BLOCKER, seed=SEED):
    gold = gold_pairs(data)
    gold_set = {frozenset(p) for p in gold}

    est = ZeroEREstimator(blocker=blocker)          # 'standard' => no PLM download
    blocks = est.build_blocks(data)
    cand = [(eid, c) for eid, cs in blocks.items() for c in cs]

    pos = [p for p in cand if frozenset(p) in gold_set]
    neg = [p for p in cand if frozenset(p) not in gold_set]
    kept = {frozenset(p) for p in pos}
    blocking_recall = len(kept) / max(len(gold_set), 1)   # upper bound on F1
    pos = pos + [p for p in gold if frozenset(p) not in kept]   # re-add dropped golds

    rng = np.random.default_rng(seed)
    if len(neg) > n_negatives:
        neg = [neg[i] for i in rng.choice(len(neg), n_negatives, replace=False)]

    pairs = pos + neg
    labels = np.concatenate([np.ones(len(pos), int), np.zeros(len(neg), int)])
    info = dict(blocking_recall=round(blocking_recall, 3), n_pos=len(pos), n_neg=len(neg))
    return pairs, labels, info

## 2. PLM embeddings

`tsne_2d` standardises, PCA-reduces high-dim PLM vectors to 50-D, then projects to 2-D. 
`manual_features` reuses the live pipeline;
`plm_pair_vectors` concatenates a record's columns, embeds, and forms `|u − v|` per pair.

In [ ]:
def tsne_2d(X, seed=SEED):
    Xs = StandardScaler().fit_transform(X)
    if Xs.shape[1] > 50:                       # de-noise high-dim PLM space first
        Xs = PCA(n_components=50, random_state=seed).fit_transform(Xs)
    perp = max(5.0, min(30.0, (len(Xs) - 1) / 3.0))
    return TSNE(n_components=2, perplexity=perp, init="pca",
                learning_rate="auto", random_state=seed).fit_transform(Xs)


def manual_features(data, attrs, pairs):
    fm = _build_feature_matrix(pairs, data, attrs)   # rows aligned with `pairs`
    return fm.values, fm.shape[1]


def plm_pair_vectors(data, attrs, pairs, mid, prefix=""):
    """Concat all attribute columns -> one text per record -> embed -> |u-v| per pair.

    ``prefix`` is prepended to every record string (required by the e5 family).
    Encoding runs on ``DEVICE`` (GPU when available); the model is loaded fresh
    and freed afterwards so the next — possibly multi-GB — model has room rather
    than every model in ``PLM_MODELS`` staying resident at once.
    """
    texts = data.entities[attrs].astype(str).agg(" ".join, axis=1).to_numpy()
    used = sorted({i for p in pairs for i in p})     # embed only referenced records
    slot = {i: k for k, i in enumerate(used)}
    model = SentenceTransformer(mid, device=DEVICE)
    batch = 128 if DEVICE == "cuda" else 64
    emb = np.asarray(model.encode([prefix + texts[i] for i in used],
                                  normalize_embeddings=True, batch_size=batch,
                                  show_progress_bar=False, device=DEVICE))
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    li = np.array([slot[a] for a, _ in pairs])
    ri = np.array([slot[b] for _, b in pairs])
    return np.abs(emb[li] - emb[ri]), emb.shape[1]

## 3. Plotting

In [4]:
def scatter(ax, coords, labels, title):
    neg, pos = coords[labels == 0], coords[labels == 1]
    ax.scatter(neg[:, 0], neg[:, 1], s=6, c="#9bb7d4", alpha=0.35,
               linewidths=0, label=f"non-match ({len(neg)})")
    ax.scatter(pos[:, 0], pos[:, 1], s=12, c="#d62728", alpha=0.8,
               linewidths=0, label=f"match ({len(pos)})")
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    ax.legend(loc="best", fontsize=7, framealpha=0.8, markerscale=1.5)

In [ ]:
def pair_records(data, attrs, pairs, labels):
    """One row per pair: label, original ids, and each attribute's left/right value."""
    inv = {v: k for k, v in data._ids_mapping_1.items()}
    inv.update({v: k for k, v in data._ids_mapping_2.items()})
    li, ri = map(np.asarray, zip(*pairs))
    cols = {
        "label":    labels.astype(int),
        "left_id":  [inv.get(int(i)) for i in li],
        "right_id": [inv.get(int(i)) for i in ri],
    }
    for a in attrs:
        col = data.entities[a].astype(str).to_numpy()
        cols[f"left_{a}"], cols[f"right_{a}"] = col[li], col[ri]
    return pd.DataFrame(cols)

## 4. Run all feature spaces per dataset

Per dataset: build the shared pair set once, then t-SNE the manual features and
every PLM `|u−v|` space.

In [ ]:
all_exports = []
for ds in DATASETS_TO_RUN:
    print(f"=== {ds} ===", flush=True)
    data = load_data(ds)
    attrs = DATASETS[ds]["attributes"]
    pairs, labels, info = build_pairs_and_labels(data)
    print(f"  {info['n_pos']} matches / {info['n_neg']} non-matches "
          f"| blocking recall {info['blocking_recall']}", flush=True)

    base = pair_records(data, attrs, pairs, labels)   # constant across methods

    methods = []                                       # (name, dim, coords, model_id)
    Xm, dim = manual_features(data, attrs, pairs)
    methods.append(("manual", dim, tsne_2d(Xm), "manual sim-features"))
    for short, (mid, prefix) in PLM_MODELS.items():
        print(f"  embedding with {short} ({mid}) on {DEVICE} ...", flush=True)
        Xp, dim = plm_pair_vectors(data, attrs, pairs, mid, prefix)
        methods.append((short, dim, tsne_2d(Xp), mid))

    # --- tidy long export: one row per (pair x method) ---------------------
    # `model_id` carries the actual HF id (or "manual sim-features") so the
    # dashboard's PLM dropdown can show real model names, not just short codes.
    frames = []
    for name, dim, coords, model_id in methods:
        f = base.copy()
        f.insert(0, "dataset", ds)
        f.insert(1, "method", name)
        f.insert(2, "model_id", model_id)
        f.insert(3, "dim", dim)
        f["x"], f["y"] = coords[:, 0], coords[:, 1]
        frames.append(f)
    export = pd.concat(frames, ignore_index=True)
    all_exports.append(export)
    pq = os.path.join(OUT_DIR, f"tsne_{ds}.parquet")
    export.to_parquet(pq, index=False)

    # --- static overview grid (adapts to N panels) ------------------------
    n = len(methods); ncols = 4; nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.2 * ncols, 3.0 * nrows),
                             squeeze=False)
    for ax, (name, dim, coords, _mid) in zip(axes.flat, methods):
        title = (f"manual sim-features ({dim}-D)" if name == "manual"
                 else f"{name}: |u-v| ({dim}-D)")
        scatter(ax, coords, labels, title)
    for ax in axes.flat[n:]:
        ax.axis("off")
    fig.suptitle(f"{ds} — t-SNE of candidate-pair space "
                 f"(blocking recall {info['blocking_recall']})", fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    png = os.path.join(OUT_DIR, f"tsne_{ds}.png")
    fig.savefig(png, dpi=150)
    plt.show()

    print(f"  saved {pq}, {png}", flush=True)

combined = os.path.join(OUT_DIR, "tsne_all.parquet")
pd.concat(all_exports, ignore_index=True).to_parquet(combined, index=False)
print("wrote combined export:", combined, flush=True)